# Análise de dados TCP-CII

### Importação dos parâmetros universais

In [75]:
import importlib.util
from pathlib import Path

path = Path("../../../parametros/config.py").resolve()

spec = importlib.util.spec_from_file_location("parametros", path)
parametros = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parametros)

In [76]:
# Parâmetros importados do arquivo config.py
print(
    "Filtrar por quantidade de alelos TCC2:.........................",
    parametros.filtarar_por_qte_de_alelos_tcc2,
)
print(
    "Parâmetro de filtragem median binding percentile TCC2:.........",
    parametros.parametro_de_filtragem_mbp_tcc2,
)
print(
    "Percentual de match mínimo TCC2:...............................",
    parametros.percent_match_minimo_tcc2,
)

Filtrar por quantidade de alelos TCC2:......................... 2
Parâmetro de filtragem median binding percentile TCC2:......... 1
Percentual de match mínimo TCC2:............................... 95.0


In [77]:
import pandas as pd

In [78]:
df = pd.read_csv('./T CELL/DENV 4 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNQTWQIEKASLIEVK,206,221,16,HLA-DRB1*01:01,246,0.01,WQIEKASLI,0.986066,0.01
1,1,KNQTWQIEKASLIEVKT,206,222,17,HLA-DRB1*01:01,314,0.04,WQIEKASLI,0.978820,0.04
2,1,KNQTWQIEKASLIEV,206,220,15,HLA-DRB1*01:01,178,0.06,WQIEKASLI,0.974156,0.06
3,1,KNQTWQIEKASLIEVKTC,206,223,18,HLA-DRB1*01:01,382,0.06,WQIEKASLI,0.961404,0.06
4,1,WIESSKNQTWQIEKASLIEVK,201,221,21,HLA-DRB1*01:01,582,0.07,WQIEKASLI,0.829909,0.07
...,...,...,...,...,...,...,...,...,...,...,...
16411,1,TTASGKLVTQWCCR,301,314,14,HLA-DRB3*02:02,129,100.00,SGKLVTQWC,0.000011,100.00
16412,1,TTASGKLVTQWCCRSCTMPP,301,320,20,HLA-DRB3*02:02,535,100.00,LVTQWCCRS,0.000011,100.00
16413,1,KLVTQWCCRSCTMPPLRFLGE,306,326,21,HLA-DRB1*04:01,603,100.00,CRSCTMPPL,0.000010,100.00
16414,1,TTASGKLVTQWCC,301,313,13,HLA-DRB3*02:02,61,100.00,TASGKLVTQ,0.000006,100.00


## Selecionando Epítopos por median binding percentile.

In [79]:
mbp_minimo = parametros.parametro_de_filtragem_mbp_tcc2

In [80]:
df_mbp_m5 = df[df['median binding percentile'] < mbp_minimo].copy()
print("Filtrando por median binding percentile < ", mbp_minimo)
df_mbp_m5

Filtrando por median binding percentile <  1


,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNQTWQIEKASLIEVK,206,221,16,HLA-DRB1*01:01,246,0.01,WQIEKASLI,0.986066,0.01
1,1,KNQTWQIEKASLIEVKT,206,222,17,HLA-DRB1*01:01,314,0.04,WQIEKASLI,0.978820,0.04
2,1,KNQTWQIEKASLIEV,206,220,15,HLA-DRB1*01:01,178,0.06,WQIEKASLI,0.974156,0.06
3,1,KNQTWQIEKASLIEVKTC,206,223,18,HLA-DRB1*01:01,382,0.06,WQIEKASLI,0.961404,0.06
4,1,WIESSKNQTWQIEKASLIEVK,201,221,21,HLA-DRB1*01:01,582,0.07,WQIEKASLI,0.829909,0.07
...,...,...,...,...,...,...,...,...,...,...,...
86,1,GSGIFVVDNVHTWTEQYKF,16,34,19,HLA-DRB3*01:01,411,0.90,FVVDNVHTW,0.349148,0.90
87,1,KNQTWQIEKASLIEVKT,206,222,17,HLA-DRB3*02:02,314,0.93,WQIEKASLI,0.454709,0.93
88,1,QKAVHADMGYWIES,191,204,14,HLA-DRB3*01:01,107,0.95,VHADMGYWI,0.432979,0.95
89,1,AAIKDQKAVHADM,186,198,13,HLA-DRB1*01:01,38,0.98,IKDQKAVHA,0.608534,0.98


## Agrupando por pepitideos e agregando colunas pertinentes

In [81]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    )
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDQKAVHADM,186,198,2,0.745,"HLA-DRB1*01:01, HLA-DRB4*01:01"
1,AAIKDQKAVHADMG,186,199,1,0.720,HLA-DRB4*01:01
2,AKIFTPEARNSTF,121,133,1,0.800,HLA-DRB1*11:01
3,ERRAWNSLEVEDY,146,158,1,0.490,HLA-DQA1*05:01/DQB1*02:01
4,ERRAWNSLEVEDYG,146,159,1,0.260,HLA-DQA1*05:01/DQB1*02:01
...,...,...,...,...,...,...
75,YRQGYATQTVGPWHLGK,256,272,1,0.520,HLA-DQA1*05:01/DQB1*02:01
76,YRQGYATQTVGPWHLGKL,256,273,1,0.510,HLA-DQA1*05:01/DQB1*02:01
77,YRQGYATQTVGPWHLGKLE,256,274,1,0.660,HLA-DQA1*05:01/DQB1*02:01
78,YRQGYATQTVGPWHLGKLEI,256,275,1,0.820,HLA-DQA1*05:01/DQB1*02:01


## Filtragem por qte_de_alelos

In [82]:
# mbp_minimo = 1
qte_alelos_minimo = parametros.filtarar_por_qte_de_alelos_tcc2

In [83]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= qte_alelos_minimo
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDQKAVHADM,186,198,2,0.745,"HLA-DRB1*01:01, HLA-DRB4*01:01"
1,HRLMSAAIKDQKA,181,193,2,0.755,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1..."
2,KNQTWQIEKASLIE,206,219,2,0.480,"HLA-DRB1*01:01, HLA-DRB1*07:01"
3,KNQTWQIEKASLIEVK,206,221,3,0.700,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB3*02:02"
4,KNQTWQIEKASLIEVKT,206,222,2,0.485,"HLA-DRB1*01:01, HLA-DRB3*02:02"
5,QYKFQPESPARLA,31,43,2,0.455,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
6,QYKFQPESPARLAS,31,44,2,0.520,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
7,QYKFQPESPARLASA,31,45,2,0.505,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
8,YRQGYATQTVGPW,256,268,2,0.465,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
9,YRQGYATQTVGPWH,256,269,2,0.385,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."


## Sorting por median_biding_percentile

In [84]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,YRQGYATQTVGPWH,256,269,2,0.385,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
1,QYKFQPESPARLA,31,43,2,0.455,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
2,YRQGYATQTVGPW,256,268,2,0.465,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
3,KNQTWQIEKASLIE,206,219,2,0.480,"HLA-DRB1*01:01, HLA-DRB1*07:01"
4,KNQTWQIEKASLIEVKT,206,222,2,0.485,"HLA-DRB1*01:01, HLA-DRB3*02:02"
5,QYKFQPESPARLASA,31,45,2,0.505,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
6,QYKFQPESPARLAS,31,44,2,0.520,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
7,KNQTWQIEKASLIEVK,206,221,3,0.700,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB3*02:02"
8,AAIKDQKAVHADM,186,198,2,0.745,"HLA-DRB1*01:01, HLA-DRB4*01:01"
9,HRLMSAAIKDQKA,181,193,2,0.755,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1..."


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [85]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos_tcell_2.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0       YRQGYATQTVGPWH
1        QYKFQPESPARLA
2        YRQGYATQTVGPW
3       KNQTWQIEKASLIE
4    KNQTWQIEKASLIEVKT
5      QYKFQPESPARLASA
6       QYKFQPESPARLAS
7     KNQTWQIEKASLIEVK
8        AAIKDQKAVHADM
9        HRLMSAAIKDQKA
Name: peptide, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [86]:
# !seqkit grep -s -v -r -p '[-*]' './Fastas/denv1_NS1_proteinas.fa' > DENV1_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [87]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_2.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,YRQGYATQTVGPWH,14,62.42% (93/149),85.71%,100.00%,NaN
1,2,NP 2,QYKFQPESPARLA,13,98.66% (147/149),84.62%,100.00%,NaN
2,3,NP 3,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,NaN
3,4,NP 4,KNQTWQIEKASLIE,14,62.42% (93/149),78.57%,100.00%,NaN
4,5,NP 5,KNQTWQIEKASLIEVKT,17,62.42% (93/149),82.35%,100.00%,NaN
5,6,NP 6,QYKFQPESPARLASA,15,98.66% (147/149),73.33%,100.00%,NaN
6,7,NP 7,QYKFQPESPARLAS,14,98.66% (147/149),78.57%,100.00%,NaN
7,8,NP 8,KNQTWQIEKASLIEVK,16,62.42% (93/149),81.25%,100.00%,NaN
8,9,NP 9,AAIKDQKAVHADM,13,95.30% (142/149),84.62%,100.00%,NaN
9,10,NP 10,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [88]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# MPB
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "median_binding_percentile"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# start
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "start"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# end
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "end"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos
0,NP 1,YRQGYATQTVGPWH,14,62.42% (93/149),85.71%,100.00%,2,0.385,256,269,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
1,NP 2,QYKFQPESPARLA,13,98.66% (147/149),84.62%,100.00%,2,0.455,31,43,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
2,NP 3,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,2,0.465,256,268,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
3,NP 4,KNQTWQIEKASLIE,14,62.42% (93/149),78.57%,100.00%,2,0.480,206,219,"HLA-DRB1*01:01, HLA-DRB1*07:01"
4,NP 5,KNQTWQIEKASLIEVKT,17,62.42% (93/149),82.35%,100.00%,2,0.485,206,222,"HLA-DRB1*01:01, HLA-DRB3*02:02"
5,NP 6,QYKFQPESPARLASA,15,98.66% (147/149),73.33%,100.00%,2,0.505,31,45,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
6,NP 7,QYKFQPESPARLAS,14,98.66% (147/149),78.57%,100.00%,2,0.520,31,44,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01"
7,NP 8,KNQTWQIEKASLIEVK,16,62.42% (93/149),81.25%,100.00%,3,0.700,206,221,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB3*02:02"
8,NP 9,AAIKDQKAVHADM,13,95.30% (142/149),84.62%,100.00%,2,0.745,186,198,"HLA-DRB1*01:01, HLA-DRB4*01:01"
9,NP 10,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,2,0.755,181,193,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1..."


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [89]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos,percent_match
0,NP 1,YRQGYATQTVGPWH,14,62.42% (93/149),85.71%,100.00%,2,0.385,256,269,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1...",62.42
1,NP 2,QYKFQPESPARLA,13,98.66% (147/149),84.62%,100.00%,2,0.455,31,43,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01",98.66
2,NP 3,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,2,0.465,256,268,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1...",62.42
3,NP 4,KNQTWQIEKASLIE,14,62.42% (93/149),78.57%,100.00%,2,0.480,206,219,"HLA-DRB1*01:01, HLA-DRB1*07:01",62.42
4,NP 5,KNQTWQIEKASLIEVKT,17,62.42% (93/149),82.35%,100.00%,2,0.485,206,222,"HLA-DRB1*01:01, HLA-DRB3*02:02",62.42
5,NP 6,QYKFQPESPARLASA,15,98.66% (147/149),73.33%,100.00%,2,0.505,31,45,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01",98.66
6,NP 7,QYKFQPESPARLAS,14,98.66% (147/149),78.57%,100.00%,2,0.520,31,44,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01",98.66
7,NP 8,KNQTWQIEKASLIEVK,16,62.42% (93/149),81.25%,100.00%,3,0.700,206,221,"HLA-DRB1*01:01, HLA-DRB1*07:01, HLA-DRB3*02:02",62.42
8,NP 9,AAIKDQKAVHADM,13,95.30% (142/149),84.62%,100.00%,2,0.745,186,198,"HLA-DRB1*01:01, HLA-DRB4*01:01",95.30
9,NP 10,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,2,0.755,181,193,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1...",95.30


### Sort e filtragem por percent_match

In [90]:
# Minimum percentage of match for T Cell prediction
# minimum_percent_match = parametros.percent_match_minimo_tcc2
minimum_percent_match = 94

In [91]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= minimum_percent_match]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,start,end,alelos,percent_match
0,NP 2,QYKFQPESPARLA,13,98.66% (147/149),84.62%,100.00%,2,0.455,31,43,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01",98.66
1,NP 6,QYKFQPESPARLASA,15,98.66% (147/149),73.33%,100.00%,2,0.505,31,45,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01",98.66
2,NP 7,QYKFQPESPARLAS,14,98.66% (147/149),78.57%,100.00%,2,0.520,31,44,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01",98.66
3,NP 9,AAIKDQKAVHADM,13,95.30% (142/149),84.62%,100.00%,2,0.745,186,198,"HLA-DRB1*01:01, HLA-DRB4*01:01",95.30
4,NP 10,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,2,0.755,181,193,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1...",95.30


### Estrutura secundária

In [92]:
def processar_horiz(caminho_horiz):
    """
    Lê o arquivo .horiz do PSIPRED e retorna um dicionário 
    contendo as strings de confiança ('conf'), predição ('pred') e sequência ('seq').
    """
    confianca = ""
    predicao = ""
    sequencia = ""
    
    with open(caminho_horiz, 'r') as f:
        for linha in f:
            linha_str = linha.strip()
            if linha_str.startswith("Conf:"):
                confianca += linha_str.split(":", 1)[1].strip()
            elif linha_str.startswith("Pred:"):
                predicao += linha_str.split(":", 1)[1].strip()
            elif linha_str.startswith("AA:"):
                sequencia += linha_str.split(":", 1)[1].strip()
                
    return {
        "conf": confianca,
        "pred": predicao,
        "seq": sequencia
    }

In [93]:
denv_horiz = processar_horiz("./PSIPRED/denv.horiz")
denv_horiz

{'conf': '9033886999928806068997561331443157059999999999999350907996397999999987438898998735856899988700057324002379954478674225842401866587179981899445994346773389888265365006998867887544783664478209978984264689864899189999999641625478422578788540014469778699553314899530147895248189988446999799579889998632125158962463421679999869996996151425760784434310502239',
 'pred': 'CEECEEECCCCEEECCEEEEEECCCCCCEEECEEECCCHHHHHHHHHHHHHHCCEEEEECCHHHHHHHHHHHHHHHHHHHHCCCCEEEEECCCEEEEECCEECCCCCCCCCCCCCCCCCCCCEECCCCCCCEEEECCCCCCCCCHHHEEECEEEEEEEEEEEECEEEEEEECCCCCCCCCHHHHHHHHCCCEEEEECCCEEEEECCCCEEEEEEEEEEEEEECCCCCCCCCCCCCCCCCCCCCCHHHCCCCCCCCCCCCCCCCCCCCCCCCCEEEEEEECCCCEEEECCCCCCCCCCCCEEECCCCCCCCEECCCCCCCCEEEECCCCCCCCEEEEECCCCCCCCEEEEECC',
 'seq': 'DMGCVASWSGKELKCGSGIFVVDNVHTWTEQYKFQPESPARLASAILNAHKDGVCGIRSTTRLENVMWKQITNELNYVLWEGGHDLTVVAGDVKGVLTKGKRALTPPVSDLKYSWKTWGKAKIFTPEARNSTFLIDGPDTSECPNERRAWNSLEVEDYGFGMFTTNIWMKFREGSSEVCDHRLMSAAIKDQKAVHADMGYWIESSKNQTWQIEKASLIEVKTCLWPKTHTLWSNGVLESQMLIPKSYAGPFSQHNYRQGYA

In [94]:
def avaliar_epitopos_com_horiz(df_epitopos, conf, pred, seq):
    """Cruza o DataFrame de epítopos com as strings de predição do .horiz já processadas."""
    resultados = []
    
    for idx, row in df_epitopos.iterrows():
        # Identifica a coluna correta do peptídeo dependendo do estágio do pipeline
        col_pep = 'peptide' if 'peptide' in df_epitopos.columns else 'Epitope sequence'
        peptideo = row[col_pep]
        start = int(row['start'])
        end = int(row['end'])
        
        # Ajuste para índice em Python (base 0)
        idx_inicio = start - 1
        idx_fim = end
        
        # Extrai os trechos correspondentes nas strings do horiz
        sub_pred = pred[idx_inicio:idx_fim]
        sub_conf = conf[idx_inicio:idx_fim]
        
        # Conta a proporção de cada estrutura secundária
        # 'H' = Hélice, 'E' = Fita Beta, 'C' ou '.' = Coil/Alça
        n_coils = sub_pred.count('C') + sub_pred.count('.')
        n_helix = sub_pred.count('H')
        n_strand = sub_pred.count('E')
        
        resultados.append({
            'Epitope': peptideo,
            'Start': start,
            'End': end,
            'Pred_SS': sub_pred,
            'Coils': n_coils,
            'Helices': n_helix,
            'Strands': n_strand
        })
        
    return pd.DataFrame(resultados)

In [95]:
conf = denv_horiz["conf"]
pred = denv_horiz["pred"]
seq = denv_horiz["seq"]

avaliacao_epitopos = avaliar_epitopos_com_horiz(conservacy_result_filtered, conf, pred, seq)
avaliacao_epitopos

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands
0,QYKFQPESPARLA,31,43,ECEEECCCHHHHH,4,5,4
1,QYKFQPESPARLASA,31,45,ECEEECCCHHHHHHH,4,7,4
2,QYKFQPESPARLAS,31,44,ECEEECCCHHHHHH,4,6,4
3,AAIKDQKAVHADM,186,198,HHHCCCEEEEECC,5,3,5
4,HRLMSAAIKDQKA,181,193,HHHHHHHHCCCEE,3,8,2


In [96]:
# Calcula o percentual de Coils (flexibilidade) para cada epítopo
avaliacao_epitopos['Percent_Coils'] = (
    avaliacao_epitopos['Coils'] / (avaliacao_epitopos['Coils'] + avaliacao_epitopos['Helices'] + avaliacao_epitopos['Strands'])
) * 100

# Ordena os epítopos do mais flexível (maior percentual de coils) para o mais rígido
epitopos_flexiveis = avaliacao_epitopos.sort_values(
    by=['Percent_Coils', 'Coils'], 
    ascending=[False, False]
).reset_index(drop=True)

epitopos_flexiveis

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils
0,AAIKDQKAVHADM,186,198,HHHCCCEEEEECC,5,3,5,38.461538
1,QYKFQPESPARLA,31,43,ECEEECCCHHHHH,4,5,4,30.769231
2,QYKFQPESPARLAS,31,44,ECEEECCCHHHHHH,4,6,4,28.571429
3,QYKFQPESPARLASA,31,45,ECEEECCCHHHHHHH,4,7,4,26.666667
4,HRLMSAAIKDQKA,181,193,HHHHHHHHCCCEE,3,8,2,23.076923


#### Filtrar epítopos que tenham pela % de sua estrutura composta por Coils

In [97]:
valor_corte_flexibilidade = parametros.corte_flexibilidade
print(f"Valor de corte de flexibilidade (Percent_Coils): {valor_corte_flexibilidade}%")

Valor de corte de flexibilidade (Percent_Coils): 35.0%


In [98]:
corte_flexibilidade = valor_corte_flexibilidade
epitopos_filtrados_flex = epitopos_flexiveis[epitopos_flexiveis['Percent_Coils'] >= corte_flexibilidade]

epitopos_filtrados_flex

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils
0,AAIKDQKAVHADM,186,198,HHHCCCEEEEECC,5,3,5,38.461538


### Merge de dataframes

In [99]:
# Seleciona apenas as colunas exclusivas de conservação (excluindo 'start' e 'end' repetidos, se houver)
colunas_para_pegar = [c for c in conservacy_result_filtered.columns if c not in ['start', 'end', 'Epitope sequence']]
colunas_para_pegar.append('Epitope sequence') # Garante que a chave está incluída

# Realiza o merge limpo
df_consolidado = pd.merge(
    epitopos_filtrados_flex,
    conservacy_result_filtered[colunas_para_pegar],
    left_on='Epitope',
    right_on='Epitope sequence',
    how='inner'
)

# Remove se sobrou a coluna 'Epitope sequence' duplicada:
if 'Epitope sequence' in df_consolidado.columns and 'Epitope' in df_consolidado.columns:
    df_consolidado = df_consolidado.drop(columns=['Epitope sequence'])

df_consolidado

,Epitope,Start,End,Pred_SS,Coils,Helices,Strands,Percent_Coils,Epitope name,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,median_binding_percentile,alelos,percent_match
0,AAIKDQKAVHADM,186,198,HHHCCCEEEEECC,5,3,5,38.461538,NP 9,13,95.30% (142/149),84.62%,100.00%,2,0.745,"HLA-DRB1*01:01, HLA-DRB4*01:01",95.3


In [100]:
epitopos_filtrados_flex.to_csv("epitopos_validados_flexibilidade_tcell2.csv", index=False)